&larr; Previous: [JN7 · The re-key and the audit](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN7_rekey_audit.ipynb)

# JN8 · The record watches itself

In JN6a you treated the city's state filing as the **answer key**, and in JN6b you graded it.
Here is the uncomfortable final lesson: **the answer key moves.** Cities amend filings; the
state's export pipeline changes; a number you verified in June may not be the number the
portal shows in October. A record verified *once* is not a record verified *forever*.

So the last skill in this course is not an analysis — it's a **habit**: pin a dated snapshot,
re-pull the live source on a schedule, and diff. Ten minutes of code turns "we checked" into
"we are checking." This notebook is the miniature version of the watcher that guards the real
Berkeley audit (its production sibling, JN-G, runs on a schedule with five error-detectors).

In [ ]:
# === COLAB BOOTSTRAP — fetch ONLY the pinned oracle snapshot from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import urllib.request

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
_here = Path.cwd()
def _repo_ok(_here):
    """True only if a scripts/ tree exists AND housing_rules actually imports from it.
    A stale Colab extraction satisfies 'the directory exists' while being unusable, which
    previously skipped both the module refetch AND the data fetch. Anything that cannot
    import is treated as absent; under /content (a disposable Colab tree, never a real
    checkout) the broken copy is removed so the fetch below replaces it."""
    import importlib, shutil
    for _base in [_here] + list(_here.parents):
        if not (_base/'scripts'/'build_v2').exists():
            continue
        sys.path.insert(0, str(_base/'scripts'))
        try:
            for _m in [k for k in list(sys.modules)
                       if k.split('.')[0] in ('housing_rules', 's0_keys', 'cpra_dedup')]:
                del sys.modules[_m]
            importlib.invalidate_caches()
            import housing_rules  # noqa: F401  - the real test: does the package satisfy its own __init__?
            return True
        except Exception as _e:
            print(f'modules present but unusable ({type(_e).__name__}: {_e}); refetching')
            try: sys.path.remove(str(_base/'scripts'))
            except ValueError: pass
            # Remove the broken tree ONLY where it is a downloaded extraction, never a real
            # checkout: a genuine repo has .git beside scripts/. Without this removal the
            # fetch below is skipped (its own guard also only tests existence) and the stale
            # copy survives — which is precisely the bug this replaces.
            if not (_base/'.git').exists():
                shutil.rmtree(_base/'scripts', ignore_errors=True)
                print('removed the unusable scripts/ tree; it will be re-downloaded')
            return False
    return False

_have_repo = _repo_ok(_here)
REPO = next((p for p in [_here, *_here.parents] if (p/'scripts'/'build_v2').exists()), _here)

def _get(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})  # r2.dev 403s python-urllib
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

PINNED = REPO/'databases'/'hcd_apr_mirror_2026-06-17_fresh.db'
if not PINNED.exists():
    PINNED.parent.mkdir(parents=True, exist_ok=True)
    PINNED.write_bytes(_get(f'{R2}/data/hcd_apr_mirror_2026-06-17_fresh.db'))
    print('fetched pinned snapshot ->', PINNED.name)
print('pinned oracle snapshot:', PINNED.name, f'({PINNED.stat().st_size:,} bytes, mirrored 2026-06-17)')

## 1 · What the filing said when we pinned it

The snapshot is a mirror of HCD's statewide APR dataset taken on a **known date** (2026-06-17,
confirmed identical to the live portal that day). We compute one summary from it: Berkeley's
completed units (certificates of occupancy) per year — the same sum JN6a taught, across the
11 income-tier CO columns of Table A2.

In [ ]:
import sqlite3

def tier_cols(colnames):
    # the 11 CO income-tier columns, found by RULE not by name-list — schemas drift
    # (the live portal has already respelled one column since our snapshot!)
    return [c for c in colnames if c.upper().startswith('CO_') and
            ('INCOME' in c.upper() or c.upper().endswith(('_DR', '_NDR')))]

def co_by_year(rows_iter):
    out, tiers = {}, None
    for rec in rows_iter:
        if tiers is None:
            tiers = tier_cols(rec.keys())
            assert len(tiers) == 11, f'expected 11 CO tier columns, found {len(tiers)}: {tiers}'
        y = str(rec['YEAR'])[:4]
        total = sum(int(float(rec[c] or 0)) for c in tiers)
        if total: out[y] = out.get(y, 0) + total
    return dict(sorted(out.items()))

con = sqlite3.connect(f'file:{PINNED}?mode=ro', uri=True)
con.row_factory = sqlite3.Row
pinned = co_by_year(dict(r) for r in con.execute("SELECT * FROM table_a2 WHERE JURIS_NAME LIKE 'BERKELEY%'"))
print('PINNED (2026-06-17) — Berkeley completed units by year:')
for y, u in pinned.items(): print(f'  {y}: {u:,}')

## 2 · What the filing says right now

Same computation, but against the **live** dataset on California's open-data portal, pulled this
minute through its API. Note the tier columns are found by *rule*, not by a hard-coded name list —
while building this notebook we discovered the live portal had already **respelled a column** since
our snapshot. Sources are alive all the way down to their schemas.

In [ ]:
import json, urllib.parse, urllib.request

RESOURCE = 'fe505d9b-8c36-42ba-ba30-08bc4f34e022'   # APR Table A2 on data.ca.gov
def live_rows():
    offset = 0
    while True:
        q = urllib.parse.urlencode({'resource_id': RESOURCE, 'limit': 500, 'offset': offset,
                                    'filters': json.dumps({'JURIS_NAME': 'BERKELEY'})})
        with urllib.request.urlopen(f'https://data.ca.gov/api/3/action/datastore_search?{q}', timeout=60) as r:
            batch = json.load(r)['result']['records']
        if not batch: return
        yield from batch
        offset += 500

live = co_by_year(live_rows())
print('LIVE (today) — Berkeley completed units by year:')
for y, u in live.items(): print(f'  {y}: {u:,}')

## 3 · The watch: has the answer key moved?

This is the whole trick — two dated summaries and a subtraction. **Read the verdict carefully
either way:** "no drift" means your verification still stands; "drift" does *not* mean fraud —
filings get legitimately amended. Drift means one thing only: *go look, and re-date your evidence.*

In [ ]:
years = sorted(set(pinned) | set(live))
drift = {y: (pinned.get(y, 0), live.get(y, 0)) for y in years if pinned.get(y, 0) != live.get(y, 0)}
if not drift:
    print('NO DRIFT — the filing has not moved since the 2026-06-17 snapshot.')
    print('Your JN6b scorecard still stands, and now you can PROVE it stands as of today.')
else:
    print(f'THE FILING MOVED — {len(drift)} year(s) changed since 2026-06-17:')
    for y, (was, now) in drift.items():
        print(f'  {y}: pinned {was:,} -> live {now:,}  ({now-was:+,})')
    print('\nWhat to do (and what NOT to do):')
    print('  1. NEVER edit your pinned snapshot to match — evidence is append-only.')
    print('  2. Save a NEW dated snapshot alongside the old one.')
    print('  3. Re-run your JN6b scorecard against the new snapshot; the DIFF of scorecards')
    print('     is itself a finding: what did the city change, and why?')

## The habit, generalized

What you just built is a **watcher**: *pinned snapshot → live pull → diff → dated verdict.*
It works on any source that can move under you — a state dataset, a city portal, an API your
newsroom relies on. The production version guarding the Berkeley audit (JN-G) adds a schedule,
an append-only log of every run, and five mechanical error-detectors over the diffs — but the
skeleton is exactly these three cells.

The course's arc, complete: **look first (JN00) → build the record (JN1–JN5) → grade the answer
key (JN6) → audit your own keys (JN7) → and never stop watching (JN8).**